In [0]:
spark.sql("USE CATALOG e_comm")
spark.sql("USE SCHEMA bronze")

In [0]:
spark.sql("create table if not exists bronze.orders(order_id string, customer_id string, order_status string, order_purchase_timestamp string, order_approved_at string, order_delivered_carrier_date string, order_delivered_customer_date string, order_estimated_delivery_date string, merge_flag boolean, ingestion_ts timestamp) using delta ")

In [0]:
from pyspark.sql.functions import lit, current_timestamp
import requests
import pandas as pd
from io import StringIO
url = "https://raw.githubusercontent.com/deepakmali17/E_commerce_dataplatform/refs/heads/main/datasets/orders.csv"

response = requests.get(url)

response.raise_for_status()
pdf  = pd.read_csv(StringIO(response.text))
df = spark.createDataFrame(pdf)
df = df.withColumn("merge_flag", lit(False)).withColumn("ingestion_ts", current_timestamp())


In [0]:
df.write.format("delta").mode("overwrite").option("mergeSchema" , True).saveAsTable("e_comm.bronze.orders")

In [0]:
%sql
select count(*) from e_comm.bronze.orders